# ACX3300 Project 1: Forecasting Earnings

**Team Members:**
- Yizhou Hu (36076406)
- Phong Phan (33867003)
- Jiaye Yuan (Student ID 3)

# Table of Contents

需要添加目录

# Overview

The purpose of the project is to forecast future firm earnings using accounting data. We examine whether past earnings information and various financial variables can be used to effectively predict future earnings performance. This is important as the forecast assists investors, analysts and stakeholders in evaluating firm performance, profitability and future value.

This project applies both time series forecasting models and accounting-based forecasting models. The time series models include random walk model and moving average models. The random walk model assumes that future earnings per share can be predicted using current earnings per share (EPS). Moving average models are also used to smooth short-term fluctuations in earnings by averaging EPS over the past 2, 3 and 5 years.

The project also includes an OLS regression model to forecast future profitability. The model examines whether current profitability and firm characteristics help explain future earnings performance.[有可能在代码更改后需要更改]

The conceptual regression model is:

$$
ROA_{t+1} = \beta_0 + \beta_1 ROA_t + \beta_2 PM_t + \beta_3 Loss_t + \beta_4 \log AT_t + \varepsilon
$$

More specifically, this regression predicts next year's ROA ($ROA_{t+1}$) using current variables observed in year $t$.

## Load & Explore Raw Data

In [ ]:
import pandas as pd
import numpy as np
from IPython.display import display

df = pd.read_csv('accounting_data.csv')
n1 = len(df)
df.head()

## Data Cleaning

描述下面这段代码的目的。

In [ ]:
essential_cols = ['gvkey', 'fyear', 'at', 'ceq', 'ni', 'sale', 'csho']
df.dropna(subset=essential_cols, inplace=True)

df.sort_values(by=['gvkey', 'fyear'], inplace=True)
n2 = len(df)

df = df[(df['at'] > 0) & (df['ceq'] > 0) & (df['sale'] > 0) & (df['csho'] != 0)]
n3 = len(df)

# Dealing with Outliers

## Winsorization

描述一下下面这段代码的目的

In [ ]:
def wins(df, vars):
    def clip_series(s):
        lower = s.quantile(0.01)
        upper = s.quantile(0.99)
        return s.clip(lower=lower, upper=upper)
    for var in vars:
        df[var] = df.groupby('fyear')[var].transform(clip_series)
    return df

cols_to_winsorize = ['at', 'ni', 'sale', 'ceq', 'csho']
df = wins(df, cols_to_winsorize).dropna()  # if remove dropna() from this line, the result will be very different
n4 = len(df)

print("Yearly Winsorization complete.")

winsor_check = df.groupby('fyear')[['at']].agg(['min', 'max']).head(10)
display(winsor_check)

## Data Preparation



The dependent variable in the model is future return on assets ($ROA_{t+1}$), which measures future profitability. It is calculated as next year's net income divided by current total assets:

$$
ROA_{t+1} = \frac{\text{Net Income}_{t+1}}{\text{Total Assets}_t}
$$

This variable is used because ROA measures how efficiently a firm generates earnings from its asset base and is commonly used as a measure of firm performance.

Current return on assets ($ROA_t$) is included as an explanatory variable because profitability is often persistent over time, meaning firms with strong current profitability may continue to perform well in future periods. It is calculated as:

$$
ROA_t = \frac{\text{Net Income}_t}{\text{Total Assets}_t}
$$

Profit margin ($PM_t$) is included because it measures operating profitability and shows how much profit a firm generates from each dollar of sales revenue. Firms with higher profit margins may have stronger and more sustainable future earnings. It is calculated as:

$$
PM_t = \frac{\text{Net Income}_t}{\text{Sales Revenue}_t}
$$

Firm size ($\log AT_t$) is included because larger firms are generally more stable and may have more predictable earnings patterns. Firm size is measured using the natural logarithm of total assets to reduce skewness caused by extremely large firms:

$$
\log AT_t = \ln(\text{Total Assets}_t)
$$

Furthermore, it aligns the model with the business reality of diminishing marginal effects. Without a logarithmic transformation, a linear model assumes that an absolute asset increase of $10 million has the exact same impact on a $10 million firm as it does on a $100 billion corporate giant. Taking the natural logarithm ensures the model evaluates proportional, rather than absolute, growth, correcting this violation of business common sense.

The loss indicator ($Loss_t$) is included to capture whether a firm reports negative earnings. Firms reporting losses may experience lower earnings persistence and greater uncertainty surrounding future profitability. The variable is constructed as:

$$
Loss_t = \begin{cases} 1 & \text{if Net Income}_t < 0 \\ 0 & \text{otherwise} \end{cases}
$$

The following code constructs the dependent and independent variables for the OLS regression model and prepares earnings per share (EPS) variables for the time series forecasting models.

Future net income ($NI_{t+1}$) is obtained by shifting the net income series one year forward within each firm using `groupby('gvkey')['ni'].shift(-1)`. The `groupby` operation ensures the shift is applied separately for each firm, preventing cross-firm data leakage. Observations without a next-year net income value — the final year for each firm — are dropped. The OLS variables ($ROA_{t+1}$, $ROA_t$, $PM_t$, $\log AT_t$, $Loss_t$) are then computed from $NI_{t+1}$ and the winsorized raw variables according to the formulas introduced in the Data Preparation section above.

For the time series forecasting models (random walk and moving average), current and future EPS values are constructed:

$$EPS_t = \frac{NI_t}{\text{CSHO}_t}$$

Future EPS values ($EPS_{t+1}$, $EPS_{t+2}$, $EPS_{t+3}$) are obtained by shifting the EPS series forward by one, two, and three periods within each firm:

$$EPS_{t+k} = \frac{NI_{t+k}}{\text{CSHO}_{t+k}} \quad \text{for } k = 1, 2, 3$$

The `groupby('gvkey')['eps'].shift(-k)` operation again maintains firm-level boundaries so that EPS values are never shifted across different companies.

After the ratios are computed, any infinite values — caused by division by zero in intermediate calculations — are replaced with `NaN` and subsequently dropped. Observations with missing values in any key variable (`roa_t1`, `pm_t`, `eps`, `eps_t1`, `eps_t2`, `eps_t3`) are removed to ensure a complete and consistent dataset for all subsequent analyses.

Finally, a `selection_summary` DataFrame is constructed to track the number of observations remaining after each filtering step, from the raw data through winsorization and variable construction. This provides a transparent audit trail of how the sample evolves from 285,491 initial observations to the final analytical sample.


In [ ]:
df['ni1'] = df.groupby('gvkey')['ni'].shift(-1)
df.dropna(subset=['ni1'], inplace=True)

df['roa_t1'] = df['ni1'] / df['at']
df['pm_t']  = df['ni'] / df['sale']
df['roa_t'] = df['ni'] / df['at']
df['log_at'] = np.log(df['at'])
df['loss_t'] = np.where(df['ni']<0, 1, 0)

df['eps'] = df['ni'] / df['csho']
df['eps_t1'] = df.groupby('gvkey')['eps'].shift(-1)
df['eps_t2'] = df.groupby('gvkey')['eps'].shift(-2)
df['eps_t3'] = df.groupby('gvkey')['eps'].shift(-3)

df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(subset=['roa_t1', 'pm_t', 'eps', 'eps_t1', 'eps_t2', 'eps_t3'], inplace=True)
n5 = len(df)

selection_summary = pd.DataFrame({
    'Filtering Step': [
        '1. Raw Data from accounting_data.csv',
        '2. Drop missing essential fields',
        '3. Economic Logic (at, ceq, sale > 0; csho != 0)',
        '4. Winsorize raw variables at 1%/99% by year',
        '5. Calculate ratios & EPS, remove Inf/NaN'
    ],
    'Observations Remaining': [n1, n2, n3, n4, n5],
    'Observations Dropped': [0, n1 - n2, n2 - n3, n3 - n4, n4 - n5]
})

selection_summary['Observations Remaining'] = selection_summary['Observations Remaining'].apply(lambda x: f'{x:,}')
selection_summary['Observations Dropped'] = selection_summary['Observations Dropped'].apply(lambda x: f'{x:,}')

display(selection_summary)

The initial dataset contained 285,491 observations obtained from the accounting dataset. Observations with missing values for essential variables including total assets (at), common equity (ceq), net income (ni), sales (sale), and shares outstanding (csho) were removed, resulting in 186,048 observations. The sample was then restricted to firms with positive total assets, positive common equity, positive sales, and non-zero shares outstanding, resulting in the sample size remaining the same at 147,228. Next, the selected financial variables were winsorized at the 1st and 99th percentiles by financial year to reduce the influence of extreme outliers. After removing any remaining missing observations, the sample consisted of 63,213 observations. Following the calculation of ROA, profit margin, EPS, and future EPS variables, observations containing missing or infinite values were excluded. The final research sample consisted of 34,614 observations

## Ratio Winsorization

In the previous data preparation step, although we winsorized the raw
financial variables — at, ni, sale, ceq, and csho — at the 1st and 99th
percentiles by fiscal year, we still observe extreme values in the ratio
variables computed from these winsorized components (roa_t1, pm_t, and
roa_t).

Specifically, describe() reveals that roa_t1 reaches a maximum of 25.09,
pm_t reaches 414.17, and roa_t reaches 24.59. By comparison, the 75th
percentiles stand at only 0.079, 0.112, and 0.071 respectively — the
maximum values exceed the upper end of the reasonable range by several
orders of magnitude. These values carry no economic meaning (a pm_t of
414 implies $414 of net income per dollar of sales revenue, which is
impossible in reality). If left untreated, they would severely distort
the subsequent Pearson correlation analysis and OLS regression estimates.

We believe these extreme ratio values are not driven by outliers in the
underlying raw variables, but rather arise from the boundary combination
effect. Winsorization guarantees that each individual variable falls within
the [1%, 99%] interval, but it does not prevent two boundary values from
producing an extreme result when combined through division. For instance,
if a firm's net income (ni) happens to lie near the 1st-percentile boundary
while its sales revenue (sale) lies near the 99th-percentile boundary, the
ratio pm_t = ni / sale can produce an artificially extreme value. The root
cause is that the numerator and denominator are trimmed independently and
asynchronously — the clipping boundaries are not coordinated across variables.

To address this issue, we apply a second round of winsorization directly to
the three ratio variables themselves.

First, a boolean switch ENABLE_RATIO_WINSOR controls whether this step is
executed, which facilitates debugging and before/after comparison. When
enabled, we call describe() on the three variables to display their
distribution prior to winsorization, making the extreme values visible.

Next, we iterate over the ratio_vars list containing roa_t1, pm_t, and
roa_t. For each variable, we apply groupby('fyear')[var].transform to
compute the 1st and 99th percentiles independently within each fiscal year,
then use clip to trim observations beyond these bounds. We compute
percentiles by fiscal year because financial ratio distributions shift with
the economic cycle — pooling across years for percentile calculation could
lead to inappropriate trimming in certain periods.

Finally, we call describe() again on the winsorized variables to verify that
the extreme values have been successfully removed.

In [ ]:
ENABLE_RATIO_WINSOR = True
if ENABLE_RATIO_WINSOR:
    print("Before:")
    from IPython.display import display
    display(df[['roa_t1','pm_t','roa_t']].describe())

    ratio_vars = ['roa_t1', 'pm_t', 'roa_t']
    for var in ratio_vars:
        df[var] = df.groupby('fyear')[var].transform(
            lambda s: s.clip(s.quantile(0.01), s.quantile(0.99))
        )

    print("After:")
    display(df[['roa_t1','pm_t','roa_t']].describe())

# Correlation Analysis

## Descriptive Statistics

In [ ]:
desc = df[['roa_t1', 'pm_t', 'eps', 'eps_t1', 'eps_t2', 'eps_t3']].describe()
display(desc)

## Time Trend of Profit Margin

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

_, ax = plt.subplots(figsize=(16, 8))
sns.lineplot(data=df, x='fyear', y='pm_t', marker='o', ax=ax)
plt.show()

## Correlation Matrices

### Pearson Correlation

In [ ]:
from scipy.stats import pearsonr

corr_vars = ['roa_t1', 'roa_t', 'pm_t', 'log_at', 'loss_t', 'eps']

rho = df[corr_vars].corr()

mask = np.triu(np.ones_like(rho, dtype=bool))
plt.figure(figsize=(10, 8))
sns.heatmap(rho, mask=mask, annot=True, cmap='Reds', fmt='.2f')
plt.title('Pearson Correlation Matrix')
plt.show()

pval = df[corr_vars].corr(
    method=lambda x, y: pearsonr(x, y)[1]
)

np.fill_diagonal(pval.values, 1)

def significance_stars(p):
    if p <= 0.001:
        return '***'
    elif p <= 0.01:
        return '**'
    elif p <= 0.05:
        return '*'
    return ''

stars = pval.map(significance_stars)

rho_stars = rho.round(2).astype(str) + stars
rho_stars = rho_stars.where(~mask, '')
display(rho_stars)

### Spearman Correlation

In [ ]:
from scipy.stats import spearmanr

corr2 = df[corr_vars].corr(method='spearman')

plt.figure(figsize=(10, 8))
sns.heatmap(corr2, mask=mask, cmap='Reds', annot=True, fmt='.2f')
plt.title('Spearman Correlation Matrix')
plt.show()

pval_spearman = df[corr_vars].corr(
    method=lambda x, y: spearmanr(x, y)[1]
)
np.fill_diagonal(pval_spearman.values, 1)

stars_spearman = pval_spearman.map(significance_stars)
corr2_stars = corr2.round(2).astype(str) + stars_spearman
corr2_stars = corr2_stars.where(~mask, '')
display(corr2_stars)

# Conceptual Models

## Time-Series Forecast Models

### Random Walk Baseline

[现在分析这个需求] 描述下面代码目的

In [ ]:
df['error_rw'] = abs(df['ni1'] - df['ni']) / df['at']
df.head()

### Multi-period RW & Moving Average

In the code below, we set out to compare two models—Random Walk (RW) and Moving Average (MA)—and see which does a better job forecasting earnings per share (EPS).

First, we need to construct the absolute forecast error variables for each model. For the random walk models, we use the abs() function to calculate the absolute difference between actual EPS one year ahead (eps_t1), two years ahead (eps_t2), and three years ahead (eps_t3) versus the current period's EPS. This gives us our variables: error_rw_1, error_rw_2, and error_rw_3.

Second, we calculate the forecast errors for the moving average models. Since we need rolling company-level calculations, we start by grouping the EPS data by firm identifier (gvkey) using groupby('gvkey'). Then we define a custom function, calc_moving_avg, which uses pandas' rolling and mean methods to compute the 2-year, 3-year, and 5-year moving averages. We then use those moving averages (ma_2, ma_3, ma_5) as the forecasts, subtract the actual EPS one year ahead (eps_t1), and take the absolute value. That produces our error variables: error_ma_2, error_ma_3, and error_ma_5.

Finally, to make sense of the output and figure out which model is more accurate, we extract the median of all the absolute errors using the median() method. In financial data, the median usually tells a more meaningful story than the mean because it isn't thrown off by extreme outliers. Once we have the medians, we plot them as a bar chart, tapping into matplotlib and seaborn's barplot functionality. This gives us a clean, visual way to compare the forecast errors across models at a glance.

In [ ]:
df['error_rw_1'] = abs(df['eps_t1'] - df['eps'])
df['error_rw_2'] = abs(df['eps_t2'] - df['eps'])
df['error_rw_3'] = abs(df['eps_t3'] - df['eps'])

grouped_eps = df.groupby('gvkey')['eps']

def calc_moving_avg(eps_series, window_size):
    return eps_series.rolling(window=window_size, min_periods=window_size).mean()

df['ma_2'] = grouped_eps.transform(calc_moving_avg, window_size=2)
df['ma_3'] = grouped_eps.transform(calc_moving_avg, window_size=3)
df['ma_5'] = grouped_eps.transform(calc_moving_avg, window_size=5)

df['error_ma_2'] = abs(df['eps_t1'] - df['ma_2'])
df['error_ma_3'] = abs(df['eps_t1'] - df['ma_3'])
df['error_ma_5'] = abs(df['eps_t1'] - df['ma_5'])

error_cols = ['error_rw_1', 'error_rw_2', 'error_rw_3', 'error_ma_2', 'error_ma_3', 'error_ma_5']
df_errors = df.dropna(subset=error_cols)

error_medians = df_errors[error_cols].median()

plt.figure(figsize=(10, 6))
sns.barplot(x=error_medians.index, y=error_medians.values, palette='viridis', hue=error_medians.index, legend=False)
plt.title('Median Absolute Forecast Error by Model')
plt.xlabel('Model')
plt.ylabel('Median Absolute Forecast Error')
plt.show()

Here are three key takeaways:

1. RW_3 has larger errors than RW_2 and RW_1

The random walk model that projects one year ahead (RW_1) gives us a much smaller average forecast error than the versions that look two or three years out (RW_2 and RW_3). Why does the accuracy drop as we push the forecast further into the future? One reason is that uncertainty in the business environment just keeps piling up over time. In the short run, a company's accounting data and its scale of operations are fairly stable. But once you stretch the horizon, more uncontrollable shocks come into play—think macro swings or shifting competitive dynamics. What this tells us is that the further you get from today, the less historical earnings can actually tell you.

2. MA models have larger errors than RW models

The moving average (MA) models actually seem to produce larger average forecast errors than the random walk (RW) models. How can a model that works with more information be less accurate than a dead-simple RW model? One possibility is that in efficient markets, a firm's current earnings already reflect the most up-to-date operating reality. Pulling out older earnings and averaging them into today's picture doesn't add explanatory power—instead, it introduces stale noise. That noise pulls the forecast away from the firm's true earnings benchmark.

3. MA_5 has larger errors than MA_3 and MA_2

Zooming in on the moving average windows, we can also see that MA_5 generates noticeably larger forecast errors than MA_3 and MA_2. In other words, stretching the calculation window out to five years didn't make the model more accurate—it just added more error. Really old accounting data (think profits from five years ago) has almost no relevance for predicting a company's future financial performance. The core insight here is that financial information decays fast. The most recent set of financials already captures all the relevant history and the current reality.

## Model Evaluation

### Sub-sample Analysis by Firm Size

To dig deeper into whether firm size affects each model's forecasting accuracy, we split the sample into two groups—Large and Small—and compared their forecast error performance across different models.

Here's how we did the grouping. First, we computed the median of total assets (`at`) for each fiscal year using `groupby('fyear').transform('median')`. We did it year by year because total asset levels tend to drift upward over time with economic growth and inflation—comparing across years directly could distort the grouping. Then we used `np.where` to label each observation: if total assets were equal to or above that year's median asset level, the firm was classified as Large; otherwise, it went into the Small group.

For the error summary, we started by using `dropna` to make sure none of the error columns or the `size_group` variable contained missing values. Then we grouped by `size_group` and applied `median()` to compute the median absolute forecast error for each model separately for the Large and Small groups. We went with the median over the mean here because financial data can still harbor some extreme values. The median is a robust statistic—it doesn't get pulled around by outliers—so it gives a truer picture of the typical forecast error.

Finally, we reshaped the summary results into long format with `melt()` and plotted a grouped bar chart using `sns.barplot`. The x-axis shows the model names, the y-axis shows the median absolute error, and the hue separates the Large and Small groups. This gives us a clean visual comparison of how forecast errors differ between firms of different sizes.

In [ ]:
median_at_per_year = df.groupby('fyear')['at'].transform('median')
df['size_group'] = np.where(df['at'] >= median_at_per_year, 'Large', 'Small')

all_error_cols = ['error_rw_1', 'error_rw_2', 'error_rw_3',
                  'error_ma_2', 'error_ma_3', 'error_ma_5']
sub_df = df.dropna(subset=all_error_cols + ['size_group'])

median_errors = sub_df.groupby('size_group')[all_error_cols].median()

median_errors_reset = median_errors.reset_index().melt(
    id_vars='size_group', var_name='Model', value_name='Absolute Error'
)

plt.figure(figsize=(10, 6))
sns.barplot(data=median_errors_reset, x='Model', y='Absolute Error',
            hue='size_group', palette='Set2')
plt.title('Forecast Error Comparison: Large vs Small Firms (All Models)')
plt.xlabel('Model')
plt.ylabel('Median Absolute Forecast Error')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

What the grouped bar chart shows is that the absolute forecast errors are higher for the Large group than for the Small group—and this holds across the board, from the random walk models (RW_1 through RW_3) to the moving average models (MA_2 through MA_5). But the real story here isn't that the models do a worse job predicting for large firms. It's simply a scale effect baked into the absolute error metric itself.

Absolute error is calculated as `eps_t1 - eps`, and its unit is the same as EPS itself—dollars per share. Large firms tend to have much higher EPS in absolute terms.A large firm might have EPS of $5, while a small firm might have EPS of $0.50. Even if both firms have the same 10% forecast error in relative terms, the large firm ends up with an absolute error of $0.50 ($5 × 10%), while the small firm clocks in at just $0.05 ($0.50 × 10%). The large firm's error will always appear bigger. So raw absolute error, without any standardization, is not a fair for comparing forecasting accuracy across firms of different sizes.

### Scaled Forecast Error Analysis

To get around the scale problem with absolute errors, we need an error metric that doesn't depend on firm size—something that lets us compare forecasting accuracy fairly across large and small firms. So we bring in the scaled (standardized) error, calculated as:

`scaled_error = abs(eps_t1 - eps) / abs(eps)`

What this formula does is take the absolute forecast error and divide it by the absolute value of current-period EPS. The result is the error expressed as a fraction of current earnings—a relative error. 

On the implementation side, we compute the `scaled_error` for each of the three random walk models and three moving average models. To handle division-by-zero issues where EPS is near or exactly zero, we use `replace([inf, -inf], np.nan)` to swap out any infinities for missing values. After that, we clean up the data with `dropna`, group by `size_group`, and compute the median scaled error for each group. We then visualize the results using the same melt + barplot approach as before.

In [ ]:
df['scaled_error_rw_1'] = abs(df['eps_t1'] - df['eps']) / abs(df['eps'])
df['scaled_error_rw_2'] = abs(df['eps_t2'] - df['eps']) / abs(df['eps'])
df['scaled_error_rw_3'] = abs(df['eps_t3'] - df['eps']) / abs(df['eps'])

df['scaled_error_ma_2'] = abs(df['eps_t1'] - df['ma_2']) / abs(df['eps'])
df['scaled_error_ma_3'] = abs(df['eps_t1'] - df['ma_3']) / abs(df['eps'])
df['scaled_error_ma_5'] = abs(df['eps_t1'] - df['ma_5']) / abs(df['eps'])

scaled_cols = ['scaled_error_rw_1', 'scaled_error_rw_2', 'scaled_error_rw_3', 
               'scaled_error_ma_2', 'scaled_error_ma_3', 'scaled_error_ma_5']
df[scaled_cols] = df[scaled_cols].replace([np.inf, -np.inf], np.nan)

sub_df_scaled = df.dropna(subset=scaled_cols + ['size_group'])

median_scaled_errors = sub_df_scaled.groupby('size_group')[scaled_cols].median()

median_scaled_errors_reset = median_scaled_errors.reset_index().melt(
    id_vars='size_group', var_name='Model', value_name='Scaled Absolute Error'
)

plt.figure(figsize=(10, 6))
sns.barplot(data=median_scaled_errors_reset, x='Model', y='Scaled Absolute Error',
            hue='size_group', palette='Set2')
plt.title('Forecast Error Comparison: Large vs Small Firms (Scaled/Standardized)')
plt.xlabel('Model')
plt.ylabel('Median Scaled Absolute Forecast Error')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

The bar chart for scaled errors paints a completely different picture from the absolute errors. Across all six forecasting models, the Large group shows noticeably lower relative forecast errors than the Small group. What this reveals is that, once you standardize for scale, large-company earnings are actually more predictable. We analyzed three possible causes of this problem:

First, earnings persistence. Large firms tend to be mature businesses with stable market shares, diversified revenue streams, and well-established operating models. Their core profitability holds fairly steady from year to year. Current EPS simply has more explanatory power for future EPS, so history-based forecasting models naturally perform better on large firms.

Second, differences in the information environment. Large firms generally attract more analyst coverage and market scrutiny, face stricter disclosure requirements, and have less room for noise or accrual manipulation in their reported earnings. Higher-quality earnings information means current-period data reflects the underlying fundamentals more faithfully, which in turn helps improve forecast accuracy.

Third, the higher volatility of small firms. Compared to large firms, small firms are far more vulnerable to external shocks—losing a single customer, shifts in a product lifecycle, financing constraints—all of which can cause EPS to swing wildly from year to year. Even if the absolute dollar change in EPS is small, the proportional swing relative to a tiny EPS base is often large, and that directly drives up their scaled forecast errors.

To sum it up, the standardized error analysis shows that, on a relative basis, forecasting models actually produce smaller errors for large firms. That's driven by their stronger earnings persistence, better information environment, and more stable business characteristics.

# Regression Analysis

## OLS Regression

### Interpret the coefficients

We'll use the following model for the regression:

$$
ROA_{t+1} = \beta_0 + \beta_1 ROA_t + \beta_2 PM_t + \beta_3 Loss_t + \beta_4 \log AT_t + \varepsilon
$$


### Understanding the output from regressions

In [ ]:
import statsmodels.formula.api as smf

formula_hybrid = "roa_t1 ~ roa_t + pm_t + loss_t + log_at"
model_hybrid = smf.ols(formula_hybrid, data=df).fit()
print("  Hybrid Model: roa_t1 ~ roa_t + pm_t + loss_t + log_at")
model_hybrid.summary()

# Exploring Data

## Forecasting ROA and Earnings

In [ ]:
df["pred_hybrid"]  = model_hybrid.predict(df)

df["error_hybrid"]  = abs(df["roa_t1"] - df["pred_hybrid"])

error_full = pd.DataFrame({
    "Model":     ["Hybrid", "Random Walk"],
    "Median AE": [df["error_hybrid"].median(),
                  df["error_rw"].median()],
    "Mean AE":   [df["error_hybrid"].mean(),
                  df["error_rw"].mean()]
})
display(error_full)

## Comparing the forecast errors

In [ ]:
pre_2010  = df[df["fyear"] < 2010]
post_2010 = df[df["fyear"] >= 2010]

print(f"Pre-2010 observations:  {len(pre_2010):,}")
print(f"Post-2010 observations: {len(post_2010):,}")

period_errors = pd.DataFrame({
    "Model":     ["Hybrid", "Random Walk",
                  "Hybrid", "Random Walk"],
    "Period":    ["Pre-2010", "Pre-2010",
                  "Post-2010", "Post-2010"],
    "Median AE": [
        pre_2010["error_hybrid"].median(),
        pre_2010["error_rw"].median(),
        post_2010["error_hybrid"].median(),
        post_2010["error_rw"].median()
    ]
})

### Model Comparison Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.barplot(data=error_full, x="Model", y="Median AE",
            ax=axes[0], palette="Set2", hue="Model", legend=False)
axes[0].set_title("Median Absolute Error (Full Sample)")
axes[0].set_ylabel("Median AE")
axes[0].tick_params(axis="x", rotation=15)

sns.barplot(data=period_errors, x="Model", y="Median AE",
            hue="Period", ax=axes[1], palette="Set2")
axes[1].set_title("Median Absolute Error by Period")
axes[1].set_ylabel("Median AE")
axes[1].tick_params(axis="x", rotation=15)

plt.tight_layout()
plt.show()

# AI Statement

AI tools were used to assist with code structure, debugging, and markdown explanations. All outputs were verified for accuracy, economic reasoning, and compliance with project requirements.